<h1><center> corrélation groupby

<h2> 1)- corrélation

<h3> a)- Pearson : Entre deux variables numériques

In [5]:
import numpy as np


x = [1, 2, 3, 4, 8]
y = [2, 4, 6, 8, 10]  



In [8]:

r = np.corrcoef(x, y)[0, 1]
print("Coefficient de corrélation:", r)

Coefficient de corrélation: [[1.         0.93632918]
 [0.93632918 1.        ]]


In [7]:
import pandas as pd

data = pd.DataFrame({
            "x":[1, 2, 3, 4, 8] ,
            "y":[2, 4, 6, 8, 10]  
})

print(data.corr())

          x         y
x  1.000000  0.936329
y  0.936329  1.000000


<h3> b)- Test du Chi2 : Entre deux variables catégorielles

In [9]:
import seaborn as sns

titanic = sns.load_dataset('titanic')
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


1)- calculer la table de contingence 

In [19]:
titanic.groupby(['sex', 'survived']).agg({"survived": 'count'})

survived
sex    survived          
female 0               81
       1              233
male   0              468
       1              109

2)- ajout des fréquences attendues sous l'hypothese d'independance

Mais avant il faillait caluer total selon ligne et colonne puis total genrale

In [20]:
# Total ligne 1
print("Total ligne 1: 468 + 109 = ", 468 + 109)

# Total ligne 2
print("Total ligne 2: 81 + 233 = ", 81 + 233)

# Total colonne 1
print("Total colonne 1: 468 + 81 = ", 468 + 81)

# Total colonne 2
print("Total ligne 1: 109 + 233 = ", 109 + 233)



Total ligne 1: 468 + 109 =  577
Total ligne 2: 81 + 233 =  314
Total colonne 1: 468 + 81 =  549
Total ligne 1: 109 + 233 =  342


In [21]:
# Total Genral
print("Total ligne 1: 577 + 314 = ", 577 + 314)
print("Total ligne 1: 549 + 342 = ", 549 + 342)

Total ligne 1: 577 + 314 =  891
Total ligne 1: 549 + 342 =  891


Maintenant on calcule les frequences attendues

<b> Eij = [ (total ligne i) * (total ligne j )] / total général

In [25]:
E11 = (577* 549)/891
E12 = (577* 342)/891
E21 = (314* 549)/891
E22 = (314* 342)/891


print("E11= ", E11)
print("E12= ", E12)
print("E21= ", E21)
print("E22= ", E22)

E11=  355.52525252525254
E12=  221.4747474747475
E21=  193.4747474747475
E22=  120.52525252525253


3)- On calcule la statistique de test


<b>X^2 = la somme [(Oij -Eij) ^2 ]/ Eij 

</b> avec Oij est l'observation

### Meme code mais avec scipy

In [24]:
# Table de contingence
from scipy.stats import chi2_contingency

contingency = pd.crosstab(titanic['sex'], titanic['survived'])

print(contingency)

survived    0    1
sex               
female     81  233
male      468  109


In [27]:

chi2, p, dof, expected = chi2_contingency(contingency)

print("Chi² =", chi2)
print("p-value =", p)
print("Degrés de liberté =", dof)
print("expected = ", expected)

Chi² = 260.71702016732104
p-value = 1.1973570627755645e-58
Degrés de liberté = 1
expected =  [[193.47474747 120.52525253]
 [355.52525253 221.47474747]]


In [28]:
alpha = 0.05

if p < alpha:
    print("👉 Rejeter H0 (il existe une dépendance entre les variables)")
else:
    print("👉 Ne pas rejeter H0 (pas de preuve de dépendance)")

👉 Rejeter H0 (il existe une dépendance entre les variables)


### On va refaire cela pour d'autres variables 'survived' et 'who'

In [30]:
titanic.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.25,S,Third,man,True,NaN,Southampton,no,False


In [31]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(titanic['survived'], titanic['who'])
contingency

who,child,man,woman
survived,,,
0,34,449,66
1,49,88,205


In [33]:
chi2, p_value, ddl, Eij = chi2_contingency(contingency)

print("chi2 = ", chi2)
print("p_value = ", p_value)
print("ddl = ", ddl)
print("Eij = ", Eij)

chi2 =  283.923050324233
p_value =  2.2227620817798914e-62
ddl =  2
Eij =  [[ 51.14141414 330.87878788 166.97979798]
 [ 31.85858586 206.12121212 104.02020202]]


In [34]:
alpha = 0.05
if p< alpha:
    print(" survived et who sont dépendants (relation significative)")

else:
    print(" Pas de relation significative entre survived et who")

 survived et who sont dépendants (relation significative)


<h3> c)- Anova : Compare la moyenne d'une variable numérique entre plusieurs groupes catégorielles

<h1><center> The END